<a href="https://colab.research.google.com/github/Haso001/PG_ML_AI/blob/main/es-llm-finetune-Hasan-Dev/notebooks/es_training_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ES-LLM: Layer-wise Fine-Tuning mit Evolutionary Strategies

Dieses Notebook führt dich durch den kompletten Workflow:
1. **Setup** – Repo klonen, Dependencies installieren
2. **Inspect** – Modell-Architektur inspizieren
3. **Baseline** – Unveränderte Accuracy messen
4. **Train** – ES-Training auf ausgewählten Layern
5. **Evaluate** – Fine-tuned Modell testen
6. **Export** – Ergebnisse nach Google Drive speichern

**GPU-Empfehlung:** Runtime → Change runtime type → **T4 GPU** (kostenlos) oder **A100** (Colab Pro)

## 0. GPU prüfen

In [15]:
import torch
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Keine GPU! Gehe zu Runtime → Change runtime type → GPU")

CUDA verfügbar: True
GPU: Tesla T4
VRAM: 15.6 GB


## 1. Setup – Repo klonen & Dependencies installieren

In [16]:
# ── Repo klonen ──
import os

REPO_URL = "https://github.com/Haso001/PG_ML_AI.git"
REPO_DIR = "/content/PG_ML_AI"
PROJECT_DIR = f"{REPO_DIR}/es-llm-finetune-Hasan-Dev"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} existiert bereits.")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

/content/PG_ML_AI existiert bereits.
Working directory: /content/PG_ML_AI/es-llm-finetune-Hasan-Dev


In [17]:
# ── Alternativ: Repo als ZIP hochladen ──
# Falls du kein Git-Repo hast, lade den Ordner als ZIP hoch:
#
# from google.colab import files
# uploaded = files.upload()  # ZIP-Datei hochladen
# !unzip -o es-llm-finetune.zip -d /content/es-llm-finetune
# os.chdir("/content/es-llm-finetune")

In [18]:
# ── Dependencies installieren ──
!pip install -q torch transformers datasets accelerate pyyaml cmaes

# Verifizieren
import torch, transformers
print(f"torch={torch.__version__}, transformers={transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

torch=2.10.0+cu128, transformers=5.0.0
CUDA: True


In [19]:
# ── src/ zum Python-Pfad hinzufügen ──
import sys
sys.path.insert(0, f"{PROJECT_DIR}/src")

# Import-Test
from es_llm.model.loader import load_model_and_tokenizer, inspect_model_layers
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.training.es_trainer import train
from es_llm.utils.config import load_config
print("Alle Imports erfolgreich!")

Alle Imports erfolgreich!


## 2. Modell-Architektur inspizieren

Zuerst schauen wir uns an, welche Layer das Qwen2.5-0.5B hat und wie viele Parameter pro Layer existieren.

In [20]:
# Modell laden (float16 auf GPU → ~1 GB VRAM)
model, tokenizer = load_model_and_tokenizer(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="float16",
    device="cuda",
)

total = sum(p.numel() for p in model.parameters())
print(f"\nGesamt: {total:,} Parameter ({total/1e6:.1f}M)")
print(f"VRAM belegt: {torch.cuda.memory_allocated()/1e9:.2f} GB")

[20:16:06] es_llm INFO: Loading model: Qwen/Qwen2.5-0.5B-Instruct  (dtype=float16, device=cuda)


INFO:es_llm:Loading model: Qwen/Qwen2.5-0.5B-Instruct  (dtype=float16, device=cuda)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[20:16:12] es_llm INFO: Model loaded – 494.0M parameters


INFO:es_llm:Model loaded – 494.0M parameters



Gesamt: 494,032,768 Parameter (494.0M)
VRAM belegt: 3.06 GB


In [21]:
# Alle Layer auflisten
layers = inspect_model_layers(model)

# Per-Layer Zusammenfassung
from collections import defaultdict
per_layer = defaultdict(int)
for l in layers:
    parts = l["name"].split(".")
    key = "other"
    for i, p in enumerate(parts):
        if p == "layers" and i + 1 < len(parts) and parts[i + 1].isdigit():
            key = f"layers.{parts[i + 1]}"
            break
    per_layer[key] += l["numel"]

print(f"{'Layer':<20s} {'Parameter':>15s}")
print("─" * 37)
for key in sorted(per_layer, key=lambda k: (not k.startswith("layers"), k)):
    print(f"{key:<20s} {per_layer[key]:>15,}")

Layer                      Parameter
─────────────────────────────────────
layers.0                  14,912,384
layers.1                  14,912,384
layers.10                 14,912,384
layers.11                 14,912,384
layers.12                 14,912,384
layers.13                 14,912,384
layers.14                 14,912,384
layers.15                 14,912,384
layers.16                 14,912,384
layers.17                 14,912,384
layers.18                 14,912,384
layers.19                 14,912,384
layers.2                  14,912,384
layers.20                 14,912,384
layers.21                 14,912,384
layers.22                 14,912,384
layers.23                 14,912,384
layers.3                  14,912,384
layers.4                  14,912,384
layers.5                  14,912,384
layers.6                  14,912,384
layers.7                  14,912,384
layers.8                  14,912,384
layers.9                  14,912,384
other                    136,135,552


In [22]:
# Detail: Layer 23 (letzter Decoder-Block)
print("\nLayer 23 – Detail:")
print(f"{'Name':<55s} {'Shape':<25s} {'Params':>10s}")
print("─" * 92)
for l in layers:
    if "layers.23" in l["name"]:
        print(f"{l['name']:<55s} {str(l['shape']):<25s} {l['numel']:>10,}")


Layer 23 – Detail:
Name                                                    Shape                         Params
────────────────────────────────────────────────────────────────────────────────────────────
model.layers.23.self_attn.q_proj.weight                 [896, 896]                   802,816
model.layers.23.self_attn.q_proj.bias                   [896]                            896
model.layers.23.self_attn.k_proj.weight                 [128, 896]                   114,688
model.layers.23.self_attn.k_proj.bias                   [128]                            128
model.layers.23.self_attn.v_proj.weight                 [128, 896]                   114,688
model.layers.23.self_attn.v_proj.bias                   [128]                            128
model.layers.23.self_attn.o_proj.weight                 [896, 896]                   802,816
model.layers.23.mlp.gate_proj.weight                    [4864, 896]                4,358,144
model.layers.23.mlp.up_proj.weight                

## 3. Layer-Selektion testen

Bevor wir trainieren, prüfen wir, welche Parameter der `LayerSelector` auswählt.

In [23]:
# Nur LayerNorm des letzten Layers → wenige Parameter, ideal zum Starten
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="layernorm",
)
print(selector.summary())

[20:16:12] es_llm INFO: LayerSelector: 2 target parameters selected (1,792 total elements)


INFO:es_llm:LayerSelector: 2 target parameters selected (1,792 total elements)


Strategy: by_index  |  Components: layernorm
Total target params: 2  (1,792 elements)

  model.layers.23.input_layernorm.weight                        shape=[896]  numel=       896
  model.layers.23.post_attention_layernorm.weight               shape=[896]  numel=       896


In [24]:
# MLP des letzten Layers → mehr Parameter
selector_mlp = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="mlp",
)
print(selector_mlp.summary())

[20:16:12] es_llm INFO: LayerSelector: 3 target parameters selected (13,074,432 total elements)


INFO:es_llm:LayerSelector: 3 target parameters selected (13,074,432 total elements)


Strategy: by_index  |  Components: mlp
Total target params: 3  (13,074,432 elements)

  model.layers.23.mlp.down_proj.weight                          shape=[896, 4864]  numel= 4,358,144
  model.layers.23.mlp.gate_proj.weight                          shape=[4864, 896]  numel= 4,358,144
  model.layers.23.mlp.up_proj.weight                            shape=[4864, 896]  numel= 4,358,144


In [25]:
# Attention Q/K/V der letzten 2 Layer
selector_attn = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[22, 23],
    components="attention_qkv",
)
print(selector_attn.summary())

[20:16:12] es_llm INFO: LayerSelector: 12 target parameters selected (2,066,688 total elements)


INFO:es_llm:LayerSelector: 12 target parameters selected (2,066,688 total elements)


Strategy: by_index  |  Components: attention_qkv
Total target params: 12  (2,066,688 elements)

  model.layers.22.self_attn.k_proj.bias                         shape=[128]  numel=       128
  model.layers.22.self_attn.k_proj.weight                       shape=[128, 896]  numel=   114,688
  model.layers.22.self_attn.q_proj.bias                         shape=[896]  numel=       896
  model.layers.22.self_attn.q_proj.weight                       shape=[896, 896]  numel=   802,816
  model.layers.22.self_attn.v_proj.bias                         shape=[128]  numel=       128
  model.layers.22.self_attn.v_proj.weight                       shape=[128, 896]  numel=   114,688
  model.layers.23.self_attn.k_proj.bias                         shape=[128]  numel=       128
  model.layers.23.self_attn.k_proj.weight                       shape=[128, 896]  numel=   114,688
  model.layers.23.self_attn.q_proj.bias                         shape=[896]  numel=       896
  model.layers.23.self_attn.q_proj.wei

## 4. Baseline messen

Wie gut ist das Modell **unverändert** auf GSM8K?

In [26]:
# Baseline-Fitness (kleines Subset zum schnellen Testen)
fitness_fn = GSM8KFitness(
    num_samples=30,       # Anzahl Fragen
    split="test",
    max_new_tokens=256,
)

baseline_acc = fitness_fn.evaluate(model, tokenizer)
print(f"\n✅ Baseline Accuracy (GSM8K test, n=30): {baseline_acc:.2%}")

[20:16:12] es_llm INFO: Loading GSM8K dataset (split=test, n=30)...


INFO:es_llm:Loading GSM8K dataset (split=test, n=30)...


[20:16:13] es_llm INFO: GSM8K loaded: 30 examples


INFO:es_llm:GSM8K loaded: 30 examples



✅ Baseline Accuracy (GSM8K test, n=30): 26.67%


## 5. ES-Training starten

### Option A: Mit YAML-Config (empfohlen)

In [27]:
# Config laden und starten
cfg = load_config(f"{PROJECT_DIR}/configs/gsm8k_last_layer.yaml")

# Config anzeigen
import json
print(json.dumps(cfg, indent=2))

{
  "model": {
    "name": "Qwen/Qwen2.5-0.5B-Instruct",
    "revision": "main",
    "dtype": "float16",
    "device": "cuda"
  },
  "layer_selection": {
    "strategy": "by_index",
    "layer_indices": [
      23
    ],
    "components": "all"
  },
  "es": {
    "algorithm": "openai_es",
    "population_size": 20,
    "sigma": 0.01,
    "learning_rate": 0.001,
    "num_generations": 100,
    "seed": 42,
    "antithetic": true,
    "fitness_shaping": "centered_rank",
    "weight_decay": 0.0,
    "low_rank": {
      "enabled": false,
      "rank": 16
    }
  },
  "fitness": {
    "task": "gsm8k",
    "split": "train",
    "num_samples": 30,
    "max_new_tokens": 256,
    "batch_size": 1
  },
  "output": {
    "dir": "experiments/runs",
    "run_name": "gsm8k_last_layer_es",
    "save_every": 25,
    "log_every": 1
  },
  "eval": {
    "task": "gsm8k",
    "split": "test",
    "num_samples": 200,
    "max_new_tokens": 256
  }
}


In [28]:
# Optional: Werte überschreiben
cfg["es"]["num_generations"] = 50        # Schneller Testlauf
cfg["es"]["population_size"] = 20
cfg["fitness"]["num_samples"] = 30       # Samples pro Fitness-Eval
cfg["output"]["dir"] = "/content/experiments/runs"

# ACHTUNG: Bei Option A wird das Modell NEU geladen.
# Vorheriges Modell freigeben, um VRAM zu sparen:
del model, tokenizer
torch.cuda.empty_cache()

# Training starten!
run_dir = train(cfg)
print(f"\n✅ Training abgeschlossen! Ergebnisse in: {run_dir}")

[20:19:49] es_llm INFO: Run directory: /content/experiments/runs/gsm8k_last_layer_es


INFO:es_llm:Run directory: /content/experiments/runs/gsm8k_last_layer_es


[20:19:49] es_llm INFO: Loading model: Qwen/Qwen2.5-0.5B-Instruct  (dtype=float16, device=cuda)


INFO:es_llm:Loading model: Qwen/Qwen2.5-0.5B-Instruct  (dtype=float16, device=cuda)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

[20:19:55] es_llm INFO: GSM8K loaded: 30 examples


INFO:es_llm:GSM8K loaded: 30 examples


[20:23:36] es_llm INFO: Baseline fitness (gsm8k_train_30): 0.1667


INFO:es_llm:Baseline fitness (gsm8k_train_30): 0.1667


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

### Option B: Manueller Training-Loop (mehr Kontrolle)

Falls du den Loop selbst steuern willst (z.B. um live zu plotten):

In [ ]:
# ── Manueller ES-Loop ──
# (Nur ausführen wenn du Option A NICHT genutzt hast)

import torch
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],       # Letzter Layer
    components="layernorm",   # Nur LayerNorm (~1.8K Params)
)
print(f"Target: {selector.num_target_elements:,} Parameter")

# 3) ES-Algorithmus
es = OpenAIES(
    population_size=20,
    sigma=0.01,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
)

# 4) Fitness-Funktion
fitness_fn = GSM8KFitness(num_samples=20, split="train", max_new_tokens=256)

# 5) Training
current_params = selector.get_flat_params().clone()
history = []

NUM_GENERATIONS = 30

for gen in range(1, NUM_GENERATIONS + 1):
    # Ask
    candidates = es.ask(current_params)

    # Evaluate
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)

    # Tell
    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    history.append({
        "gen": gen,
        "best": result.best_fitness,
        "mean": result.mean_fitness,
    })
    print(f"Gen {gen:3d} | best={result.best_fitness:.3f}  mean={result.mean_fitness:.3f}")

## 6. Ergebnisse visualisieren

In [ ]:
import matplotlib.pyplot as plt

# Falls Option A → history aus training_log.jsonl laden:
# import json
# with open(run_dir / "training_log.jsonl") as f:
#     history = [json.loads(l) for l in f]

gens = [h.get("gen") or h.get("generation") for h in history]
bests = [h.get("best") or h.get("best_fitness") for h in history]
means = [h.get("mean") or h.get("mean_fitness") for h in history]

plt.figure(figsize=(10, 5))
plt.plot(gens, bests, label="Best Fitness", linewidth=2)
plt.plot(gens, means, label="Mean Fitness", alpha=0.7)
plt.axhline(y=baseline_acc, color="red", linestyle="--", label="Baseline")
plt.xlabel("Generation")
plt.ylabel("Accuracy (GSM8K)")
plt.title("ES Training – Fitness über Generationen")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Fine-tuned Modell evaluieren

In [ ]:
# Evaluation auf dem Test-Split (größeres Subset)
eval_fitness = GSM8KFitness(
    num_samples=100,
    split="test",
    max_new_tokens=256,
)

final_acc = eval_fitness.evaluate(model, tokenizer)
print(f"\n📊 Ergebnisse:")
print(f"   Baseline (test, n=30):   {baseline_acc:.2%}")
print(f"   Fine-tuned (test, n=100): {final_acc:.2%}")
print(f"   Differenz:                {final_acc - baseline_acc:+.2%}")

## 8. Modell & Ergebnisse speichern

In [ ]:
# ── Checkpoint speichern ──
import json
from pathlib import Path

save_dir = Path("/content/experiments/final")
save_dir.mkdir(parents=True, exist_ok=True)

# Layer-Gewichte speichern
layer_state = {name: p.data.cpu().clone() for name, p in selector.get_target_params()}
torch.save(layer_state, save_dir / "best_layer_weights.pt")

# Komplettes Modell speichern
model.save_pretrained(save_dir / "model")
tokenizer.save_pretrained(save_dir / "model")

# Ergebnisse speichern
results = {
    "baseline_accuracy": baseline_acc,
    "final_accuracy": final_acc,
    "target_layers": selector.target_names,
    "num_target_params": selector.num_target_elements,
    "history": history,
}
(save_dir / "results.json").write_text(json.dumps(results, indent=2, default=str))

print(f"✅ Gespeichert in: {save_dir}")
!ls -la {save_dir}

In [ ]:
# ── Optional: Nach Google Drive kopieren ──
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/es-llm-experiments"
!mkdir -p {DRIVE_DIR}
!cp -r {save_dir}/* {DRIVE_DIR}/
print(f"✅ Kopiert nach Google Drive: {DRIVE_DIR}")

## 9. Gespeichertes Modell laden (in neuer Session)

So lädst du ein zuvor gespeichertes ES-Modell:

In [ ]:
# ── Variante A: Komplettes gespeichertes Modell ──
# model, tokenizer = load_model_and_tokenizer(
#     model_name="/content/experiments/final/model",
#     dtype="float16",
#     device="cuda",
# )

# ── Variante B: Base-Modell + Layer-Gewichte ──
# model, tokenizer = load_model_and_tokenizer(
#     "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
# )
# state = torch.load("/content/experiments/final/best_layer_weights.pt", weights_only=True)
# model_dict = dict(model.named_parameters())
# for name, tensor in state.items():
#     model_dict[name].data.copy_(tensor.to(model.device))
# print(f"Loaded {len(state)} layer parameters")

---

## Tipps für Colab

| Thema | Empfehlung |
|-------|------------|
| **GPU-Typ** | T4 (free) reicht für Qwen 0.5B. A100 (Pro) für größere Modelle |
| **dtype** | Immer `float16` auf GPU → halber Speicher, schneller |
| **VRAM** | Qwen 0.5B in fp16 ≈ 1 GB. T4 hat 16 GB → viel Headroom |
| **Timeout** | Colab trennt nach ~90min Inaktivität. Halte Tab offen |
| **Ergebnisse** | Immer nach Google Drive speichern – Colab ist flüchtig! |
| **Batched Eval** | `num_samples=50-100` ist ein guter Kompromiss |
| **population_size** | 20-50 auf T4, 50-100 auf A100 |

### Empfohlene Experiment-Reihenfolge

1. **LayerNorm-only** (Schritt 1) – ~1.8K Params, schnell, proof-of-concept
2. **attention_qkv** eines Layers – ~1.1M Params
3. **MLP eines Layers** – ~13M Params, ggf. mit Low-Rank
4. **Mehrere Layer** – Layer 20-23, MLP+attention